# AI for the Classroom

A subset of NRP's GPUs power a community-shared, **OpenAI-compatible LLM endpoint** at `https://ellm.nrp-nautilus.io/v1`. No pod to run, no GPU to hold, no per-seat license.

This notebook is a short taste of two things a class gets for free on this hub:

1. **An LLM you can call from Python** — the token is already in your environment.
2. **Jupyter AI** — an assistant living in JupyterLab that can explain and fix the code in front of you.

> 📘 [Managed LLMs](https://nrp.ai/documentation/userdocs/ai/llm-managed/) · [Available models](https://nrp.ai/documentation/userdocs/ai/llm-managed/models/) · [Get your own token](https://nrp.ai/llmtoken)

## 1. The token is already here

On this training hub, `OPENAI_API_BASE` and `OPENAI_API_KEY` are exported into every server for you — **students never handle a key**. On your own hub you would mint one at [nrp.ai/llmtoken](https://nrp.ai/llmtoken) and set the same two variables.

In [ ]:
import os

print("base:", os.environ.get("OPENAI_API_BASE", "(not set)"))
key = os.environ.get("OPENAI_API_KEY", "")
print("key: ", (key[:8] + "…") if key else "(not set)")

## 2. Ask it something

The `openai` SDK talks to NRP unchanged — only `base_url` differs from the commercial API.

One wrinkle worth knowing: the SDK looks for `OPENAI_BASE_URL`, but NRP exports `OPENAI_API_BASE`, so pass it explicitly or the client quietly calls `api.openai.com` instead.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=os.environ["OPENAI_API_BASE"])

resp = client.chat.completions.create(
    model="gpt-oss",
    messages=[
        {"role": "user",
         "content": "In two sentences, what is the National Research Platform?"},
    ],
)

print(resp.choices[0].message.content)

**That is the whole trick.** The same eight lines work against a commercial API, against NRP, or against a vLLM server you run yourself on a GPU pod — you change `base_url` and nothing else. Teach the OpenAI-compatible API once and the skill outlives whatever model is fashionable.

Other models are a string away — `qwen3`, `gemma`, `kimi`, `minimax-m2`, `deepseek-v4-flash` and more:

In [ ]:
for m in client.models.list().data:
    print(m.id)

## 3. Jupyter AI

This hub ships [Jupyter AI](https://jupyter-ai.readthedocs.io/) already pointed at the NRP endpoint — no install, no key to paste.

- **Chat panel:** click the **chat icon** in the left sidebar and ask a question.
- **In a cell:** load the magics and prefix a cell with `%%ai`.

```python
%load_ext jupyter_ai_magics
```

```text
%%ai openai-chat:gpt-oss
Explain what a Kubernetes namespace is, for someone who has never used one.
```

### Now break something on purpose

The three cells below are broken. Run them, then hand the error to Jupyter AI — paste the traceback into the chat panel and ask it what went wrong.

This is the demo worth showing a class: the assistant is *in the notebook*, looking at the same code the student is stuck on.

In [ ]:
# 🐞 Cell 1 — run me
temperatures_c = [18, 21, 25, 30, 12]

def to_fahrenheit(c):
    return c * 9 / 5 + 32

for t in temperatures_c:
    print(t, "C =", to_farenheit(t), "F")

In [ ]:
# 🐞 Cell 2 — run me
student_scores = {"ana": 88, "ben": 92, "cleo": 79}

total = 0
for name in student_scores:
    total += name

print("class average:", total / len(student_scores))

In [ ]:
# 🐞 Cell 3 — run me. This one does NOT crash. It is still wrong.
readings = [3, 7, 2, 9, 4, 8]

# intended: the average of every reading after the first
average = sum(readings[1:]) / len(readings)

print("average of readings 2..6 =", average)
print("expected:", (7 + 2 + 9 + 4 + 8) / 5)

Cell 3 is the interesting one. It runs, prints a number, and the number is wrong — the kind of bug that quietly survives into a student's homework. Ask Jupyter AI *"why do these two numbers differ?"* and watch it reason about the code rather than just read a traceback.

## What to take away

- The endpoint is **OpenAI-compatible**, so every tool that speaks that API — notebooks, Jupyter AI, agents, your own scripts — works against NRP with a `base_url` change.
- On a hub you run, the token is an environment variable *you* set once, so **no student ever handles a credential**.
- An assistant sitting inside JupyterLab changes what a stuck student does at 2am.

Next: [deploy a JupyterHub of your own](3_custom_jupyterhub.html), with the image menu, resource limits and shared storage a course needs.